In [1]:
# IMPORTANT: RUN THIS CELL IN ORDER TO IMPORT YOUR KAGGLE DATA SOURCES,
# THEN FEEL FREE TO DELETE THIS CELL.
# NOTE: THIS NOTEBOOK ENVIRONMENT DIFFERS FROM KAGGLE'S PYTHON
# ENVIRONMENT SO THERE MAY BE MISSING LIBRARIES USED BY YOUR
# NOTEBOOK.
import kagglehub
noodulz_pokemon_dataset_1000_path = kagglehub.dataset_download('noodulz/pokemon-dataset-1000')

print('Data source import complete.')


100%|██████████| 785M/785M [00:08<00:00, 101MB/s]

Extracting files...


Data source import complete.


### Neccessary Imports

In [2]:
import torch.nn as nn
import torch
from torch.utils.data import Dataset, DataLoader
from torchvision.transforms import transforms
from PIL import Image
import pandas as pd
from pathlib import Path
import os

### Path

In [11]:
import kagglehub
path = kagglehub.dataset_download("noodulz/pokemon-dataset-1000")

Using Colab cache for faster access to the 'pokemon-dataset-1000' dataset.


In [12]:
train_data_path = Path("/kaggle/input/pokemon-dataset-1000/pokemon-dataset-1000/train")
test_data_path = Path("/kaggle/input/pokemon-dataset-1000/pokemon-dataset-1000/test")


In [13]:
img_width = 64
img_height = 64

### Transformation

In [14]:
train_transform = transforms.Compose(transforms=
                                    [
                                        transforms.Resize((img_width, img_height)),
                                        transforms.RandomHorizontalFlip(0.5),
                                        transforms.RandomVerticalFlip(0.5),
                                        transforms.RandomRotation(degrees=30),
                                        transforms.ToTensor(),
                                        transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225])
                                    ])

test_transform = transforms.Compose(transforms=
                                    [
                                        transforms.Resize((img_width, img_height)),
                                        transforms.ToTensor(),
                                        transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225])
                                    ])


### CustomDataClass

In [15]:
class CustomDataClass(Dataset):
    def __init__(self, dir_path, transforms = None):
        self.dir_path = dir_path
        self.transforms = transforms
        self.classes = sorted(os.listdir(dir_path))
        self.img_path = []
        self.img_labels = []
        self.class_names = sorted([d.name for d in self.dir_path.iterdir() if d.is_dir()])

        for label, class_names in enumerate(self.class_names):
            class_dir = self.dir_path/class_names
            for img_file in class_dir.iterdir():
                if img_file.suffix.lower() in {".jpg", ".jpeg", ".png"}:
                    self.img_path.append(str(img_file))
                    self.img_labels.append(label)
            print(f"✅ Loaded {len(self.img_path)} images from {len(self.class_names)} classes")

    def __len__(self):
        return len(self.img_path)

    def __getitem__(self,index):
        image = Image.open(self.img_path[index]).convert("RGB")
        if self.transforms:
            image = self.transforms(image)
        return image, self.img_labels[index]

### Collate Function

In [16]:
def collate_fn(batch):
    image, label = zip(*batch)
    image = torch.stack(image)
    label = torch.tensor(label)
    return image, label


In [17]:
train_data = CustomDataClass(train_data_path, train_transform)
test_data = CustomDataClass(test_data_path, test_transform)

train_loader = DataLoader(train_data, batch_size=32, collate_fn=collate_fn, shuffle=True)
test_loader = DataLoader(test_data, batch_size=32, collate_fn=collate_fn, shuffle=False)

✅ Loaded 32 images from 1000 classes
✅ Loaded 64 images from 1000 classes
✅ Loaded 94 images from 1000 classes
✅ Loaded 110 images from 1000 classes
✅ Loaded 122 images from 1000 classes
✅ Loaded 154 images from 1000 classes
✅ Loaded 184 images from 1000 classes
✅ Loaded 216 images from 1000 classes
✅ Loaded 248 images from 1000 classes
✅ Loaded 258 images from 1000 classes
✅ Loaded 274 images from 1000 classes
✅ Loaded 304 images from 1000 classes
✅ Loaded 316 images from 1000 classes
✅ Loaded 348 images from 1000 classes
✅ Loaded 364 images from 1000 classes
✅ Loaded 396 images from 1000 classes
✅ Loaded 400 images from 1000 classes
✅ Loaded 430 images from 1000 classes
✅ Loaded 440 images from 1000 classes
✅ Loaded 450 images from 1000 classes
✅ Loaded 462 images from 1000 classes
✅ Loaded 494 images from 1000 classes
✅ Loaded 498 images from 1000 classes
✅ Loaded 530 images from 1000 classes
✅ Loaded 555 images from 1000 classes
✅ Loaded 571 images from 1000 classes
✅ Loaded 587 im

### PokeDex Model

In [18]:
class Pokedex_model(nn.Module):
    def __init__(self, num_classes):
        super().__init__()

        self.features = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=3, padding=1),
            nn.LeakyReLU(),
            nn.BatchNorm2d(32),
            nn.MaxPool2d(2, padding=1),

            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.LeakyReLU(),
            nn.BatchNorm2d(64),
            nn.MaxPool2d(2, padding=1),

            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.LeakyReLU(),
            nn.BatchNorm2d(128),
            nn.MaxPool2d(2, padding=1),

            nn.Conv2d(128, 256, kernel_size=3, padding=1),
            nn.LeakyReLU(),
            nn.BatchNorm2d(256),
            nn.MaxPool2d(2, padding=1)
        )

        self.classifier = nn.Sequential(
            nn.AdaptiveAvgPool2d((1,1)),
            nn.Flatten(),
            nn.Dropout(0.4),
            nn.Linear(256, num_classes)
        )


    def forward(self, x):

        x = self.features(x)
        x = self.classifier(x)

        return x


In [19]:
num_classes = len(train_data.classes)
device = "cuda" if torch.cuda.is_available() else "cpu"
loss = nn.CrossEntropyLoss()
model = Pokedex_model(num_classes).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr = 0.001)
model


Pokedex_model(
  (features): Sequential(
    (0): Conv2d(3, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): LeakyReLU(negative_slope=0.01)
    (2): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (3): MaxPool2d(kernel_size=2, stride=2, padding=1, dilation=1, ceil_mode=False)
    (4): Conv2d(32, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (5): LeakyReLU(negative_slope=0.01)
    (6): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (7): MaxPool2d(kernel_size=2, stride=2, padding=1, dilation=1, ceil_mode=False)
    (8): Conv2d(64, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (9): LeakyReLU(negative_slope=0.01)
    (10): BatchNorm2d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (11): MaxPool2d(kernel_size=2, stride=2, padding=1, dilation=1, ceil_mode=False)
    (12): Conv2d(128, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (13): Leak

In [20]:
Epochs = 10

### Training Function

In [21]:
def train(dataloader, epochs, model):
    model = model
    running_loss = 0
    for epoch in range(epochs):
        model.train()
        for _, (x,y) in enumerate(dataloader):
            x, y = x.to(device), y.to(device)
            optimizer.zero_grad()
            yhat = model(x)
            criterion = loss(yhat, y)
            criterion.backward()
            optimizer.step()
            running_loss+=criterion.item()
            print(f"Epoch {epoch}/{epochs} | Loss {criterion.item()}")

    torch.save(model.state_dict(), "pokedex_model.pth")
    print("✅ Model is Saved")
    return f"Total number of Loss is - {round(running_loss,2)}"


In [22]:
train(train_loader, Epochs, model)

Streaming output truncated to the last 5000 lines.
Epoch 2/10 | Loss 4.360132217407227
Epoch 2/10 | Loss 4.60380220413208
Epoch 2/10 | Loss 4.0596160888671875
Epoch 2/10 | Loss 4.484048366546631
Epoch 2/10 | Loss 3.638373374938965
Epoch 2/10 | Loss 4.098989009857178
Epoch 2/10 | Loss 4.054828643798828
Epoch 2/10 | Loss 4.571170806884766
Epoch 2/10 | Loss 4.668466567993164
Epoch 2/10 | Loss 4.167483329772949
Epoch 2/10 | Loss 4.11110782623291
Epoch 2/10 | Loss 4.430157661437988
Epoch 2/10 | Loss 4.478236198425293
Epoch 2/10 | Loss 3.9508140087127686
Epoch 2/10 | Loss 4.577922344207764
Epoch 2/10 | Loss 3.9808568954467773
Epoch 2/10 | Loss 4.141543388366699
Epoch 2/10 | Loss 4.216219902038574
Epoch 2/10 | Loss 4.216092586517334
Epoch 2/10 | Loss 4.707424163818359
Epoch 2/10 | Loss 4.073297023773193
Epoch 2/10 | Loss 3.9286274909973145
Epoch 2/10 | Loss 4.6431989669799805
Epoch 2/10 | Loss 4.483884811401367
Epoch 2/10 | Loss 4.211704254150391
Epoch 2/10 | Loss 4.130206108093262
Epoch 2/10

'Total number of Loss is - 20657.77'